# TruthLens image: multi-generator training

Trains ConvNeXt-Tiny + DCT on six GenImage generators. Run A holds out Midjourney and BigGAN as unseen generators. Run B also trains on Midjourney and is the one exported for serving.

In [ ]:
import time; START = time.time()


In [ ]:
REPO = 'https://github.com/Shashank17singh/TruthLens.git'
BRANCH = 'main'
import os

# Kaggle's "Save & Run All" runs headless for up to 12 hours with no idle timeout; Colab free stops after about 90 idle minutes.
KAGGLE = os.path.exists('/kaggle/working')
HOME = '/tmp/TruthLens' if KAGGLE else '/content/TruthLens'  # /tmp on Kaggle: /kaggle/working is capped at 20 GB
# Training stops cleanly after the epoch that would overrun this many minutes; rerunning it resumes.
BUDGET = 300 if KAGGLE else 75
if not os.path.exists(HOME):
    !git clone -q --branch {BRANCH} {REPO} {HOME}
%cd {HOME}
!git pull -q
assert os.path.exists('ml/image/train.py'), 'This branch does not have ml/image/train.py. Push your commits, then run this cell again.'

In [ ]:
!pip install -q -r ml/requirements.txt
import torch

GPU = torch.cuda.is_available()
print(torch.cuda.get_device_name(0) if GPU else 'No GPU. Runtime, Change runtime type, T4 GPU.')

In [ ]:
if KAGGLE:
    DRIVE = '/kaggle/working/truthlens'  # saved as the notebook's output; download it from the Output tab
else:
    from google.colab import drive, userdata

    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/truthlens'
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
import glob
import random
import shutil

assert KAGGLE, 'This notebook links the mounted Kaggle datasets.'
RAW = 'data/raw/genimage'
PER_CLASS = 5000       # per training generator, for ai and for nature
UNSEEN_FAKES = 6000    # Midjourney and BigGAN fakes from the official val split
IMAGE_SUFFIXES = ('.png', '.jpg', '.jpeg', '.webp')


def link_mounted(generator, split_dir, classes, count):
    """Link a seeded subset of one generator folder; the generator folder name becomes the label."""
    for cls in classes:
        folders = [d for depth in range(2, 7) for d in glob.glob('/kaggle/input/' + '*/' * depth + f'{split_dir}/{cls}')
                   if f'/{generator}/' in d.lower() and 'tiny-genimage' not in d]
        assert folders, f'{generator} {split_dir}/{cls} not found under /kaggle/input: attach the dataset'
        pool = sorted(os.path.join(folders[0], n) for n in os.listdir(folders[0]) if n.lower().endswith(IMAGE_SUFFIXES))
        for path in random.Random(1337).sample(pool, min(count, len(pool))):
            parts = path.split('/')
            target = os.path.join(RAW, generator, *parts[[p.lower() for p in parts].index(generator) + 1:])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            if not os.path.exists(target):
                os.symlink(path, target)
        print(generator, cls, min(count, len(pool)), 'of', len(pool), 'from', folders[0], flush=True)


for generator in ['adm', 'stable_diffusion_v_1_4', 'stable_diffusion_v_1_5', 'glide', 'wukong', 'vqdm']:
    link_mounted(generator, 'train', ['ai', 'nature'], PER_CLASS)
for generator in ['midjourney', 'biggan']:
    link_mounted(generator, 'val', ['ai'], UNSEEN_FAKES)
for generator in sorted(os.listdir(RAW)):
    for cls in ('ai', 'nature'):
        print(generator, cls, len(glob.glob(f'{RAW}/{generator}/**/{cls}/*', recursive=True)))

In [ ]:
# Resizes every image and writes data/manifests/genimage.csv with grouped 70/15/15 splits.
# ml/common/datasets.yaml holds out midjourney and biggan, so their fakes go to unseen_test only.
!python -m ml.common.prepare genimage --workers 4
from pathlib import Path
import pandas as pd
from ml.common.manifest import check_no_leakage, read_manifest, write_manifest
frame = read_manifest(Path('data/manifests/genimage.csv'))
print(frame.groupby(['split', 'generator_name']).size().to_string())

# Run B: give Midjourney fakes a seeded 70/15/15 split. Each fake is its own group, so nothing leaks; BigGAN stays unseen.
is_mj = frame['generator_name'] == 'midjourney'
frame.loc[is_mj, 'split'] = random.Random(1337).choices(['train', 'val', 'test'], weights=[70, 15, 15], k=int(is_mj.sum()))
check_no_leakage(frame)
write_manifest(frame, Path('data/manifests/genimage_mj.csv'))
print(frame[is_mj]['split'].value_counts().to_string())

In [ ]:
import subprocess, time
CKPT = '/tmp/checkpoints/image'
assert GPU
BASE = f'checkpoint_dir={CKPT} tracker=none num_workers=4 steps_per_epoch=1500 max_minutes=120 model=convnext_tiny use_frequency=true'
RUNS = [
    'manifest=data/manifests/genimage.csv version=2.0.0-heldout',
    'manifest=data/manifests/genimage_mj.csv version=2.0.0',
]
for run in RUNS:
    print('=== ', run, f'{(time.time() - START) / 3600:.2f} h', flush=True)
    subprocess.run(f'python -m ml.image.train {BASE} {run}', shell=True)
    for best in sorted(glob.glob(f'{CKPT}/*/*-best.pt')):
        name = os.path.basename(os.path.dirname(best))
        if not os.path.exists(f'{DRIVE}/results/image/{name}/metrics.json'):
            subprocess.run(f'python -m ml.image.evaluate {best} --output {DRIVE}/results/image/{name}', shell=True)
            print(open(f'{CKPT}/{name}/metrics.jsonl').read(), flush=True)
            if name.endswith('-2.0.0'):
                subprocess.run(f'python -m ml.image.export {best} --output-dir {DRIVE}/onnx', shell=True)
                os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
                shutil.copy(best, f'{DRIVE}/checkpoints/')  # serving needs the .pt for Grad-CAM
shutil.copy('data/manifests/genimage.csv', f'{DRIVE}/genimage.csv')